In [1]:
# 1) auto‑reload your .py changes
%load_ext autoreload
%autoreload 2

In [2]:
from fine_tuning.data_loader.atmos_util import standardize_atmospheric_vars
from fine_tuning.data_loader.batch_util import make_aurora_batch
from fine_tuning.data_loader.coord_util import process_lat_lon
from fine_tuning.data_loader.crop_batch_mul4 import crop_batch_to_multiple_of_4
from fine_tuning.data_loader.load_wrf import load_and_combine_wrf_files
# from fine_tuning.data_loader.pressure_level_utils import setup_50level_normalization
from fine_tuning.data_loader.pressure_util import process_pressure_levels

/home/user/anaconda3/envs/aurora2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os


DATA_DIR   = "/home/user/Documents/aurora/data_wrf"

meta, surf, static, atom = load_and_combine_wrf_files(
    os.path.join(DATA_DIR, "2d", "wrf2d_d01_2015-12-01_00:00:00.nc"),
    os.path.join(DATA_DIR, "2d", "wrf2d_d01_2015-12-01_03:00:00.nc"),
    os.path.join(DATA_DIR, "3d", "wrf3d_d01_2015-12-01_00:00:00.nc"),
    os.path.join(DATA_DIR, "3d", "wrf3d_d01_2015-12-01_03:00:00.nc"),
    os.path.join(DATA_DIR, "static", "wrfconstants_conusII.nc"),
)

static shapes: {'z': (1419, 1429), 'slt': (1419, 1429), 'lsm': (1419, 1429)}


In [4]:
# Diagnostics: raw meta / surf / static / atom
print("=== AFTER LOAD ===")
print("meta:")
for k, v in meta.items():
    print(
        f"  {k:15s}: type={type(v).__name__}, shape={getattr(v,'shape',None)}, dtype={getattr(v,'dtype',None)}"
    )
print("\nsurf_vars:")
for k, v in surf.items():
    print(f"  {k:15s}: shape={v.shape}, dtype={v.dtype}")
print("\nstatic_vars:")
for k, v in static.items():
    print(f"  {k:15s}: shape={v.shape}, dtype={v.dtype}")
print("\natom_vars:")
for k, v in atom.items():
    print(f"  {k:15s}: shape={v.shape}, dtype={v.dtype}")

=== AFTER LOAD ===
meta:
  lat            : type=ndarray, shape=(1419,), dtype=float32
  lon            : type=ndarray, shape=(1429,), dtype=float32
  time           : type=ndarray, shape=(2,), dtype=|S19
  pressure_levels: type=ndarray, shape=(50, 1419, 1429), dtype=float32

surf_vars:
  t2             : shape=(2, 1419, 1429), dtype=float32
  u10            : shape=(2, 1419, 1429), dtype=float32
  v10            : shape=(2, 1419, 1429), dtype=float32
  psfc           : shape=(2, 1419, 1429), dtype=float32

static_vars:
  z              : shape=(1419, 1429), dtype=float32
  slt            : shape=(1419, 1429), dtype=float32
  lsm            : shape=(1419, 1429), dtype=float32

atom_vars:
  z              : shape=(2, 51, 1419, 1429), dtype=float32
  t              : shape=(2, 50, 1419, 1429), dtype=float32
  u              : shape=(2, 50, 1419, 1430), dtype=float32
  v              : shape=(2, 50, 1420, 1429), dtype=float32
  q              : shape=(2, 50, 1419, 1429), dtype=float32


In [5]:

from wrf import interplevel
print("wrf-python import successful!")

wrf-python import successful!


In [8]:
# Debug 
import xarray as xr

ds = xr.open_dataset(os.path.join(DATA_DIR, "3d", "wrf3d_d01_2015-12-01_00:00:00.nc"), engine="netcdf4", decode_times=False)
p = ds["P"]

print("P dims:", p.dims)
print("P shape:", p.shape)

P dims: ('Time', 'bottom_top', 'south_north', 'west_east')
P shape: (1, 50, 1419, 1429)


In [7]:
# Debug
import xarray as xr
from wrf import destagger

ds = xr.open_dataset(os.path.join(DATA_DIR, "3d", "wrf3d_d01_2015-12-01_00:00:00.nc"))
da_u = ds["U"]            # should have dims e.g. ('Time','bottom_top_stag','south_north','west_east_stag')
print("Before destagger U dims:", da_u.dims)

# Ensure Time axis first
if da_u.ndim == 3:
    da_u = da_u.expand_dims("Time")

# Call destagger on the stag dim
da_u2 = destagger(da_u, "west_east_stag")
print("After destagger U dims:", da_u2.dims)


Before destagger U dims: ('Time', 'bottom_top', 'south_north', 'west_east_stag')


TypeError: tuple indices must be integers or slices, not str

In [6]:
from fine_tuning.data_loader.adjust_plevel import interpolate_wrf3d_to_levels
import os

DATA_DIR = "/home/user/Documents/aurora/data_wrf"
files_3d = [
    os.path.join(DATA_DIR, "3d", "wrf3d_d01_2015-12-01_00:00:00.nc"),
    os.path.join(DATA_DIR, "3d", "wrf3d_d01_2015-12-01_03:00:00.nc"),
]

# Define the 13 standard pressure levels in hPa
std_plevs = [50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000]

atom_at_levels = interpolate_wrf3d_to_levels(files_3d, std_plevs)

for var, arr in atom_at_levels.items():
    print(f"{var:>3s} → shape {arr.shape}")

/home/user/Documents/aurora/aurora_foked/aurora/fine_tuning/data_loader/adjust_plevel.py:50: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  ntime = ds.dims["Time"]
/home/user/Documents/aurora/aurora_foked/aurora/fine_tuning/data_loader/adjust_plevel.py:52: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  ny, nx = ds.dims["south_north"], ds.dims["west_east"]


ValueError: arguments 0 and 1 must have the same shape

In [25]:
# 2) Fix coords
lat1d, lon1d = process_lat_lon(meta["lat"], meta["lon"])
meta["lat"], meta["lon"] = lat1d, lon1d

In [ ]:
# Diagnostics: coords
print("\n=== AFTER COORDS PROCESSING ===")
print(
    f"lat1d.shape: {lat1d.shape}, decreasing? {all(lat1d[i]>lat1d[i+1] for i in range(len(lat1d)-1))}"
)
print(
    f"lon1d.shape: {lon1d.shape}, increasing? {all(lon1d[i]<lon1d[i+1] for i in range(len(lon1d)-1))}"
)

In [27]:
# 3) Process pressure levels
meta["pressure_levels"] = process_pressure_levels(meta["pressure_levels"], target=50)

In [ ]:
# Diagnostics: pressure levels
pl = meta["pressure_levels"]
print("\n=== AFTER PRESSURE LEVELS ===")
print(f"pressure_levels: type={type(pl).__name__}, length={len(pl)}, min={pl[0]}, max={pl[-1]}")

In [29]:
# 4) Standardize atmos shapes
H, W = surf["t2"].shape[1:]
atom = standardize_atmospheric_vars(atom, target_levels=50, target_y=H, target_x=W)

In [ ]:
# Diagnostics: standardized atmos
print("\n=== AFTER ATMOS STANDARDIZATION ===")
for k, v in atom.items():
    print(f"  {k:15s}: shape={v.shape}, dtype={v.dtype}")

In [31]:
# 5) Build the Aurora Batch
batch = make_aurora_batch(meta, surf, static, atom)

In [ ]:
# Diagnostics: final Batch
print("\n=== FINAL BATCH ===")
print("surf_vars:")
for k, v in batch.surf_vars.items():
    print(f"  {k:15s}: {tuple(v.shape)}, dtype={v.dtype}")
print("\nstatic_vars:")
for k, v in batch.static_vars.items():
    print(f"  {k:15s}: {tuple(v.shape)}, dtype={v.dtype}")
print("\natmos_vars:")
for k, v in batch.atmos_vars.items():
    print(f"  {k:15s}: {tuple(v.shape)}, dtype={v.dtype}")
print("\nmetadata:")
print(f"  lat:           {tuple(batch.metadata.lat.shape)}")
print(f"  lon:           {tuple(batch.metadata.lon.shape)}")
print(f"  time:          {batch.metadata.time}")
print(f"  atmos_levels:  {batch.metadata.atmos_levels}")

In [ ]:
# Then crop it to make it compatible with patch_size=4
cropped_batch = crop_batch_to_multiple_of_4(batch)

In [ ]:
# Diagnostics: Batch: After cropping
print("\n=== FINAL BATCH ===")
print("surf_vars:")
for k, v in cropped_batch.surf_vars.items():
    print(f"  {k:15s}: {tuple(v.shape)}, dtype={v.dtype}")
print("\nstatic_vars:")
for k, v in cropped_batch.static_vars.items():
    print(f"  {k:15s}: {tuple(v.shape)}, dtype={v.dtype}")
print("\natmos_vars:")
for k, v in cropped_batch.atmos_vars.items():
    print(f"  {k:15s}: {tuple(v.shape)}, dtype={v.dtype}")
print("\nmetadata:")
print(f"  lat:           {tuple(cropped_batch.metadata.lat.shape)}")
print(f"  lon:           {tuple(cropped_batch.metadata.lon.shape)}")
print(f"  time:          {cropped_batch.metadata.time}")
print(f"  atmos_levels:  {cropped_batch.metadata.atmos_levels}")

In [ ]:
# 3. Set up normalization for 50 pressure levels
batch_hpa = setup_50level_normalization(cropped_batch)

In [ ]:
# Diagnostics:Final Batch: After Pa-->hPa
print("\n=== FINAL BATCH ===")
print("surf_vars:")
for k, v in batch_hpa.surf_vars.items():
    print(f"  {k:15s}: {tuple(v.shape)}, dtype={v.dtype}")
print("\nstatic_vars:")
for k, v in batch_hpa.static_vars.items():
    print(f"  {k:15s}: {tuple(v.shape)}, dtype={v.dtype}")
print("\natmos_vars:")
for k, v in batch_hpa.atmos_vars.items():
    print(f"  {k:15s}: {tuple(v.shape)}, dtype={v.dtype}")
print("\nmetadata:")
print(f"  lat:           {tuple(batch_hpa.metadata.lat.shape)}")
print(f"  lon:           {tuple(batch_hpa.metadata.lon.shape)}")
print(f"  time:          {batch_hpa.metadata.time}")
print(f"  atmos_levels:  {batch_hpa.metadata.atmos_levels}")

In [16]:
import torch

from aurora import Aurora, AuroraSmall, rollout


# Run prediction with the normalized cropped batch
def run_prediction(batch, model_type="small", device="cuda", num_steps=1):
    """Run prediction using Aurora model."""
    # Initialize model based on type
    if model_type == "small":
        model = AuroraSmall(use_lora=False)
        ckpt = "aurora-0.25-small-pretrained.ckpt"
    else:
        model = Aurora(use_lora=False)
        ckpt = "aurora-0.25-pretrained.ckpt"

    # Set patch size to 4
    model.patch_size = 4

    # Load checkpoint and move to device
    model.load_checkpoint("microsoft/aurora", ckpt)
    model.eval()
    model = model.to(device)

    # Run prediction
    with torch.inference_mode():
        predictions = [pred.to("cpu") for pred in rollout(model, batch, steps=num_steps)]

    # Move model back to CPU to free GPU memory
    model = model.to("cpu")

    return predictions

In [ ]:
predictions = run_prediction(cropped_batch, model_type="small", device="cuda", num_steps=2)

In [21]:
import dataclasses

import torch

from aurora import Aurora, AuroraSmall


def run_prediction_with_memory_tracking(batch, model_type="small", device="cuda", num_steps=2):
    """Run prediction with careful memory management"""
    import gc

    def print_memory():
        print(f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        print(f"Reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB")
        print(f"Max allocated: {torch.cuda.max_memory_allocated()/1e9:.2f} GB\n")

    # Clear everything first
    torch.cuda.empty_cache()
    gc.collect()
    print("Initial GPU memory:")
    print_memory()

    # Initialize model
    if model_type == "small":
        model = AuroraSmall(use_lora=False)
        ckpt = "aurora-0.25-small-pretrained.ckpt"
    else:
        model = Aurora(use_lora=False)
        ckpt = "aurora-0.25-pretrained.ckpt"

    model.patch_size = 4
    model.load_checkpoint("microsoft/aurora", ckpt)
    model.eval()
    model = model.to(device)

    print("After model setup:")
    print_memory()

    # Keep batch on CPU initially
    batch = batch.type(torch.float32)
    batch = batch.crop(model.patch_size)

    predictions = []
    with torch.inference_mode(), torch.cuda.amp.autocast():
        current_batch = batch

        for step in range(num_steps):
            print(f"\nStep {step + 1}:")

            # Move batch to GPU just before prediction
            current_batch = current_batch.to(device)
            print("After moving batch to GPU:")
            print_memory()

            # Get prediction
            pred = model.forward(current_batch)
            print("After forward pass:")
            print_memory()

            # Move prediction to CPU immediately
            pred_cpu = pred.to("cpu")
            predictions.append(pred_cpu)

            # Prepare next batch on CPU
            if step < num_steps - 1:
                current_batch = dataclasses.replace(
                    pred_cpu,  # Use CPU prediction
                    surf_vars={
                        k: torch.cat([current_batch.surf_vars[k][:, 1:].cpu(), v], dim=1)
                        for k, v in pred_cpu.surf_vars.items()
                    },
                    atmos_vars={
                        k: torch.cat([current_batch.atmos_vars[k][:, 1:].cpu(), v], dim=1)
                        for k, v in pred_cpu.atmos_vars.items()
                    },
                )

            # Clear GPU memory
            del pred
            torch.cuda.empty_cache()
            gc.collect()

            print("After cleanup:")
            print_memory()

    # Clean up
    model = model.to("cpu")
    torch.cuda.empty_cache()
    gc.collect()

    print("\nFinal memory state:")
    print_memory()

    return predictions

In [ ]:
# Try with the smaller batch first
predictions = run_prediction_with_memory_tracking(cropped_batch, model_type="small", num_steps=2)

In [ ]:
# Print prediction shapes
print("\n=== PREDICTIONS ===")
for i, pred in enumerate(predictions):
    print(f"Prediction {i+1} shape: {pred.shape}")

## Diagnostics

In [ ]:
# Method 1: Using dir() to see all attributes
model = AuroraSmall(use_lora=False)
print("\n=== Model Attributes ===")
for attr in dir(model):
    if not attr.startswith("_"):  # Skip private attributes
        try:
            value = getattr(model, attr)
            if not callable(value):  # Skip methods, only show properties
                print(f"{attr}: {value}")
        except Exception:
            pass

# Method 2: If the model has a config attribute
print("\n=== Model Config ===")
print(model.config if hasattr(model, "config") else "No config attribute found")

# Method 3: Using vars() to see the instance variables
print("\n=== Instance Variables ===")
print(vars(model))

In [ ]:
def print_data_diagnostics(batch):
    """Print diagnostics about the batch data"""
    print("\nPressure Levels Diagnostics:")
    levels = list(batch.metadata.atmos_levels)
    print(f"Number of levels: {len(levels)}")
    print(f"Pressure range: {min(levels):.2f} to {max(levels):.2f}")
    print("\nFirst 5 levels:", levels[:5])
    print("Last 5 levels:", levels[-5:])

    print("\nAtmospheric Variables Ranges:")
    for var_name, var_data in batch.atmos_vars.items():
        print(f"\n{var_name}:")
        print(f"Shape: {var_data.shape}")
        print(f"Range: [{var_data.min().item():.2f}, {var_data.max().item():.2f}]")
        print(f"Mean: {var_data.mean().item():.2f}")

    print("\nSurface Variables Ranges:")
    for var_name, var_data in batch.surf_vars.items():
        print(f"\n{var_name}:")
        print(f"Shape: {var_data.shape}")
        print(f"Range: [{var_data.min().item():.2f}, {var_data.max().item():.2f}]")
        print(f"Mean: {var_data.mean().item():.2f}")


print_data_diagnostics(cropped_batch)

In [ ]:
def print_batch_info(batch):
    """Print information about batch dimensions"""
    print("\nBatch Dimensions:")
    print("Surface Variables:")
    for name, tensor in batch.surf_vars.items():
        print(f"{name}: {tensor.shape}, {tensor.dtype}")

    print("\nAtmospheric Variables:")
    for name, tensor in batch.atmos_vars.items():
        print(f"{name}: {tensor.shape}, {tensor.dtype}")

    # Calculate approximate memory usage
    total_elements = 0
    for tensor in batch.surf_vars.values():
        total_elements += tensor.numel()
    for tensor in batch.atmos_vars.values():
        total_elements += tensor.numel()

    # Assuming float32 (4 bytes per element)
    memory_gb = total_elements * 4 / (1024**3)
    print(f"\nApproximate batch memory usage: {memory_gb:.2f} GB")


# Use this before running prediction
print_batch_info(cropped_batch)

In [ ]:
def split_batch_spatial(batch, chunk_size=512):
    """
    Split a batch into smaller spatial chunks.
    Returns center chunk for testing.
    """
    # Calculate center coordinates
    h_center = batch.surf_vars["2t"].shape[2] // 2
    w_center = batch.surf_vars["2t"].shape[3] // 2

    # Calculate boundaries for center chunk
    h_start = h_center - chunk_size // 2
    h_end = h_start + chunk_size
    w_start = w_center - chunk_size // 2
    w_end = w_start + chunk_size

    # Create new batch with smaller spatial dimensions
    new_surf_vars = {}
    for name, tensor in batch.surf_vars.items():
        new_surf_vars[name] = tensor[..., h_start:h_end, w_start:w_end]

    new_atmos_vars = {}
    for name, tensor in batch.atmos_vars.items():
        new_atmos_vars[name] = tensor[..., h_start:h_end, w_start:w_end]

    # Create new batch with same metadata but smaller spatial region
    from aurora import Batch

    small_batch = Batch(
        surf_vars=new_surf_vars,
        atmos_vars=new_atmos_vars,
        static_vars=batch.static_vars,
        metadata=batch.metadata,
    )

    return small_batch


# Create a smaller test batch
test_batch = split_batch_spatial(cropped_batch, chunk_size=512)

# Print new batch info
print_batch_info(test_batch)

In [ ]:
def print_model_diagnostics(model_type="small"):
    """Print memory diagnostics for Aurora model"""
    import torch

    from aurora import Aurora, AuroraSmall

    # Initialize model
    if model_type == "small":
        model = AuroraSmall(use_lora=False)
        ckpt = "aurora-0.25-small-pretrained.ckpt"
    else:
        model = Aurora(use_lora=False)
        ckpt = "aurora-0.25-pretrained.ckpt"

    model.patch_size = 4

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # Estimate model size in memory
    model_size_gb = total_params * 4 / (1024**3)  # Assuming float32 (4 bytes)

    print(f"\nModel Diagnostics for Aurora-{model_type}:")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Approximate model size in memory: {model_size_gb:.2f} GB")

    # Load model to GPU and check memory
    print("\nGPU Memory Usage:")
    torch.cuda.empty_cache()
    before_load = torch.cuda.memory_allocated()

    model.load_checkpoint("microsoft/aurora", ckpt)
    model.eval()
    model = model.to("cuda")

    after_load = torch.cuda.memory_allocated()
    model_gpu_mem = (after_load - before_load) / 1024**3

    print(f"Model GPU memory usage: {model_gpu_mem:.2f} GB")
    print(f"Total GPU memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    print(f"Total GPU memory reserved: {torch.cuda.memory_reserved()/1024**3:.2f} GB")

    # Estimate memory needed for forward pass
    batch_size = 1
    seq_len = 2
    pressure_levels = 50
    spatial_dim = 512

    # Rough estimate of activation memory for one forward pass
    # This is a simplified estimate - actual memory usage might be different
    feature_dim = model.backbone.embed_dim if hasattr(model.backbone, "embed_dim") else 256
    estimated_activation_mem = (
        batch_size
        * seq_len
        * pressure_levels
        * (spatial_dim // 4)
        * (spatial_dim // 4)
        * feature_dim
        * 4
        / (1024**3)
    )

    print(f"\nEstimated activation memory for forward pass: {estimated_activation_mem:.2f} GB")
    print(f"Total estimated memory needed: {model_gpu_mem + estimated_activation_mem:.2f} GB")

    # Move model back to CPU and clear memory
    model = model.to("cpu")
    torch.cuda.empty_cache()


# Run diagnostics for small model
print_model_diagnostics("small")